# 05 — Agente de compras, herramientas y memoria

Un único Qwen2.5-7B-Instruct coordina nueve herramientas. Nuestro código conserva presupuesto, preferencias, carrito y cotización; valida cantidades, existencias, consentimiento y registro del pedido. El modelo no puede confirmar por sí solo: la tienda y las reglas de control verifican cada acción.


In [1]:
%pip -q install -r requirements-lab4-modular.txt

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements-lab4-modular.txt'
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src" / "lab4_asistente").is_dir():
    ROOT = next((p for p in ROOT.parents if (p / "src" / "lab4_asistente").is_dir()), ROOT)
sys.path.insert(0, str(ROOT / "src"))
print("Raíz del proyecto:", ROOT)
from IPython.display import display
from lab4_asistente.runtime import cargar_sistema


Raíz del proyecto: /home/winzcode/proyectos/ADEEP_LEARNING/asistente-compras-por-voz


## Cargar los componentes

In [3]:
COMP = cargar_sistema(cargar_llm=True)
SIS = COMP["sistema"]
print("Dispositivo:", COMP["dispositivo"])
print("Ubicación por componente:", COMP["dispositivos"])
print("Carga inicial por componente:", COMP["cargas"])
print("Herramientas disponibles:", len(__import__("lab4_asistente.herramientas", fromlist=["ESQUEMAS"]).ESQUEMAS))


GPU lista: NVIDIA GeForce RTX 5070 Ti Laptop GPU | VRAM libre: 10.8/11.9 GB


/home/winzcode/proyectos/ADEEP_LEARNING/asistente-compras-por-voz/.venv-dl-cuda/lib/python3.11/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Dispositivo: cuda:0
Ubicación por componente: {'llm': 'cuda:0', 'recuperacion': 'cpu'}
Carga inicial por componente: {'recuperacion': 6.262596159000168, 'llm': 12.050278918999538}
Herramientas disponibles: 9


## Interacción por turnos

In [4]:
SIS.reiniciar()
r1 = SIS.agente.turno("Busco una mochila para portátil negra, máximo 40 euros")
print(r1["respuesta"])
print("Herramientas:", [(o["herramienta"], o["resultado"].get("ok")) for o in r1["operaciones"]])
r2 = SIS.agente.turno("¿Qué opciones encontraste?")
print(r2["respuesta"])
print("Estado persistente:")
print(SIS.h.resumen_estado())


Encontré varias opciones para usted. ¿Le interesa la mochila Besttravel por 31.99 euros o la FUXINGYAO también por 31.99 euros? Ambas son negras y están disponibles.
Herramientas: [('buscar_productos', True)]
He encontrado la mochila Besttravel y la FUXINGYO, ambas negras y por 31.99 euros. ¿Cuál prefiere?
Estado persistente:
ESTADO ACTUAL (memoria del sistema):
- Presupuesto total maximo: no indicado euros
- Preferencias: ninguna
- Carrito: vacio
- Productos mostrados en la ultima busqueda:
  1) B07Q5S8K28 | Mochila Unisex Impermeable para Ordenador Portátil de hasta 15.6 Pulgadas, con Puerto USB, | 31.99 euros | stock 9
  2) B093NYPDX5 | FUXINGYAO Mochila Unisex para Ordenador Portátil de hasta 15.6 PulgadasMochila Portatil Mo | 31.99 euros | stock 9
  3) B075Z16J1Z | Mochila de Portátil, REYLEO Mochila Hombre con Puerto de USB para Ordenador hasta 15,6 Pul | 10.99 euros | stock 0
  4) B074KV9TT4 | Mochila de viaje de negocios, mochila para computadora portátil con puerto USB para ho

## Verificación del camino crítico de compra

In [5]:
# El agente debe presentar una cotización antes de poder registrar un pedido.
# La confirmación explícita se ejecuta y valida en código; la frase del LLM nunca basta.
print("Estado de cotización:", SIS.h.estado.cotizacion)
print("¿Cotización mostrada?:", SIS.h.estado.cotizacion_mostrada)


Estado de cotización: None
¿Cotización mostrada?: False
